# 종합 대시보드 (신규 화합물 MoA 및 세포독성 우선순위)

## 실행 순서
1. 패키지 설치 (최초 1회)
2. 대시보드 실행
3. 터미널에 출력되는 URL을 브라우저에서 열기

## 파일 구조
```
1차] MoA 데이터/
├── Dashboard/
│   ├── app.py              ← 진입점 (사이드바 + 탭 조립)
│   ├── helpers.py          ← 약물과 기전 정리 DB(desc, signal, assay)
│   ├── data_loader.py      ← 데이터 로딩 (@cache)
│   ├── models.py           ← 모델 학습/예측 (@cache)
│   ├── tab1_input.py       ← Tab 1: 신규 화합물 입력
│   ├── tab2_moa.py         ← Tab 2: MoA 후보 우선순위
│   ├── tab3_toxicity.py    ← Tab 3: 세포독성 리스크
│   ├── tab4_ontarget.py    ← Tab 4: On/Off-Target 예측
│   ├── tab5_rdview.py      ← Tab 5: 생물학적 및 R&D 관점
│   └── Streamlit.ipynb     ← 이 노트북 (실행 전용)
├── data/
│   ├── train_features.csv
│   ├── train_targets_scored.csv
│   ├── train_drug.csv
│   ├── cell_info.csv
│   └── gene_name.csv
└── model/
    └── eval_tabpfn_basic_pca256.csv  ← MoA 신뢰도 테이블
```

## 대시보드 탭 구성
| Tab | 파일 | 내용 |
|-----|------|------|
| Tab 1 | tab1_input.py | 신규 화합물 CSV 업로드 및 데이터 검증 |
| Tab 2 | tab2_moa.py | MoA 후보 우선순위 (Top-5, 후보별 검증 우선순위) |
| Tab 3 | tab3_toxicity.py | 세포독성 리스크 (g→c 예측, flag 분석) |
| Tab 4 | tab4_ontarget.py | 폐 특화 On/Off-Target 예측 |
| Tab 5 | tab5_rdview.py | 생물학적 및 연구개발(R&D) 관점 |

## 탭 수정 방법
각 탭을 수정하려면 해당 `tabN_*.py` 파일의 `render()` 함수만 편집하면 됩니다.  
`app.py`는 건드릴 필요 없습니다.

In [11]:
# 패키지 설치 (최초 1회만 실행)
#!pip install streamlit scikit-learn plotly pandas numpy --quiet

In [12]:
# 필수 파일 존재 여부 확인
from pathlib import Path
import pandas as pd

BASE = Path().resolve().parent
DATA = BASE / 'data'
MODEL = BASE / 'model'

required_files = [
    DATA / 'train_features.csv',
    DATA / 'train_targets_scored.csv',
    DATA / 'train_drug.csv',
    DATA / 'cell_info.csv',
    DATA / 'gene_name.csv',
    MODEL / 'eval_tabpfn_basic_pca256.csv',
]

all_ok = True
for f in required_files:
    status = '✅' if f.exists() else '❌ 파일 없음'
    print(f'{status}  {f.name}')
    if not f.exists():
        all_ok = False

print()
if all_ok:
    print('모든 필수 파일이 확인 -> 대시보드를 시작!')
else:
    print('일부 파일이 없음 -> data/ 또는 model/ 디렉토리를 확인 요망')

✅  train_features.csv
✅  train_targets_scored.csv
✅  train_drug.csv
✅  cell_info.csv
✅  gene_name.csv
✅  eval_tabpfn_basic_pca256.csv

모든 필수 파일이 확인 -> 대시보드를 시작!


In [13]:
# 데이터 미리보기
import pandas as pd
from pathlib import Path

BASE = Path().resolve().parent
DATA = BASE / 'data'

feat = pd.read_csv(DATA / 'train_features.csv', nrows=3)
tgt  = pd.read_csv(DATA / 'train_targets_scored.csv', nrows=3)

g_cols = [c for c in feat.columns if c.startswith('g-')]
c_cols = [c for c in feat.columns if c.startswith('c-')]

print(f'train_features: {feat.shape[0]}행 × {feat.shape[1]}열')
print(f'  g-features: {len(g_cols)}개 | c-features: {len(c_cols)}개')
print(f'  cp_time 값: {feat["cp_time"].unique()}')
print(f'  cp_dose 값: {feat["cp_dose"].unique()}')
print()
print(f'train_targets_scored: MoA 수 = {tgt.shape[1] - 1}개')
print()
print('샘플 데이터 (처음 3행, 주요 컬럼):')
display(feat[['sig_id', 'cp_type', 'cp_time', 'cp_dose', 'g-0', 'g-1', 'g-2']].head(3))

train_features: 3행 × 876열
  g-features: 772개 | c-features: 100개
  cp_time 값: [24 72 48]
  cp_dose 값: ['D1']

train_targets_scored: MoA 수 = 206개

샘플 데이터 (처음 3행, 주요 컬럼):


,sig_id,cp_type,cp_time,cp_dose,g-0,g-1,g-2
0,id_000644bb2,trt_cp,24,D1,1.0620,0.5577,-0.2479
1,id_000779bfc,trt_cp,72,D1,0.0743,0.4087,0.2991
2,id_000a6266a,trt_cp,48,D1,0.6280,0.5817,1.5540


In [14]:
# Streamlit 대시보드 실행
 # 새 터미널 창이 열리고, streamlit이 실행
 # "You can now view your Streamlit app" 메시지가 보이면, 성공!
 # 브라우저에서 http://localhost:8501 링크 클릭!

import subprocess
from pathlib import Path

dashboard_dir = Path().resolve()   # 이 노트북이 있는 Dashboard 폴더
app_path = dashboard_dir / 'app.py'

print(f'Dashboard 경로: {dashboard_dir}')
print(f'app.py 존재 여부: {app_path.exists()}')

if not app_path.exists():
    print()
    print('[오류] app.py 파일을 찾을 수 없습니다.')
    print('이 노트북과 app.py 가 같은 Dashboard 폴더에 있는지 확인하세요.')
else:
    # cwd 를 Dashboard 폴더로 지정하고 파일명만 넘겨서
    # 경로 특수문자(한국어, ]) 문제를 완전히 우회
    subprocess.Popen(
        ['streamlit', 'run', 'app.py', '--server.port', '8501'],
        cwd=str(dashboard_dir),          # Python이 내부적으로 경로 처리
        creationflags=subprocess.CREATE_NEW_CONSOLE,
    )
    print()
    print('새 터미널 창이 열렸습니다.')
    print()
    print('▶ 다음 단계:')
    print('  1. 새 터미널 창에서 Streamlit 시작 메시지를 확인하세요.')
    print('  2. 첫 실행 시 모델 학습으로 2~5분 소요될 수 있습니다.')
    print('  3. 아래 주소로 브라우저를 여세요:')
    print()
    print('         http://localhost:8501')
    print()
    print('▶ 종료: 새 터미널 창에서 Ctrl+C')

Dashboard 경로: C:\Users\kims\Desktop\1차] MoA 데이터\Dashboard
app.py 존재 여부: True

새 터미널 창이 열렸습니다.

▶ 다음 단계:
  1. 새 터미널 창에서 Streamlit 시작 메시지를 확인하세요.
  2. 첫 실행 시 모델 학습으로 2~5분 소요될 수 있습니다.
  3. 아래 주소로 브라우저를 여세요:

         http://localhost:8501

▶ 종료: 새 터미널 창에서 Ctrl+C


In [15]:
# 테스트용 신규 화합물 샘플 CSV 생성
 # Tab 1의 '파일 업로드'에서 사용 가능

import pandas as pd
from pathlib import Path

BASE = Path().resolve().parent
DATA = BASE / 'data'

feat = pd.read_csv(DATA / 'train_features.csv')

# trt_cp 샘플 중 일부를 신규 화합물 예시로 사용 (라벨 없음)
sample = feat[
    (feat['cp_type'] == 'trt_cp') &
    (feat['cp_time'] == 48) &
    (feat['cp_dose'] == 'D2')
].sample(5, random_state=42).reset_index(drop=True)

# compound_id 컬럼 추가, sig_id/cp_type 제거
sample.insert(0, 'compound_id', [f'NEW_CMPD_{i+1:03d}' for i in range(len(sample))])
sample = sample.drop(columns=['sig_id', 'cp_type'], errors='ignore')

out_path = Path().resolve() / 'sample_new_compound.csv'
sample.to_csv(out_path, index=False)
print(f'✅ 샘플 CSV 저장 완료: {out_path}')
print(f'   {len(sample)}개 샘플 | 컬럼 수: {sample.shape[1]}')
print()
print('이 파일을 대시보드 Tab 1의 파일 업로드에서 사용하세요.')
sample[['compound_id', 'cp_time', 'cp_dose', 'g-0', 'g-1', 'g-2']].head()

✅ 샘플 CSV 저장 완료: C:\Users\kims\Desktop\1차] MoA 데이터\Dashboard\sample_new_compound.csv
   5개 샘플 | 컬럼 수: 875

이 파일을 대시보드 Tab 1의 파일 업로드에서 사용하세요.


,compound_id,cp_time,cp_dose,g-0,g-1,g-2
0,NEW_CMPD_001,48,D2,-0.1939,-1.1860,-0.1784
1,NEW_CMPD_002,48,D2,3.7830,-0.2589,1.4310
2,NEW_CMPD_003,48,D2,1.1910,1.2860,-0.2048
3,NEW_CMPD_004,48,D2,6.4980,0.3252,3.3160
4,NEW_CMPD_005,48,D2,0.4811,-0.1069,0.5265
